# Resume Training from Checkpoint on Kaggle

This notebook shows how to load a checkpoint and continue training on Kaggle.

## Step 1: Setup and Install Dependencies

In [ ]:
# Install dependencies
!pip install -q torch torchaudio librosa soundfile jiwer pyyaml tensorboard

In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchaudio
from pathlib import Path
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Load Checkpoint Function (KAGGLE VERSION)

In [ ]:
def load_checkpoint_for_training_kaggle(checkpoint_path, device='cuda'):
    """
    Load checkpoint for continued training on Kaggle
    
    Args:
        checkpoint_path: Path to checkpoint (can be from input datasets)
        device: Device to load on
    
    Returns:
        Dictionary with model info and training state
    """
    
    # Kaggle paths - checkpoints are usually in input datasets
    kaggle_paths = [
        f"/kaggle/input/your-checkpoint-dataset/{checkpoint_path}",  # Replace with your dataset name
        f"/kaggle/working/{checkpoint_path}",
        checkpoint_path  # Direct path
    ]
    
    # Find the checkpoint
    actual_path = None
    for path in kaggle_paths:
        if os.path.exists(path):
            actual_path = path
            break
    
    if actual_path is None:
        raise FileNotFoundError(f"Checkpoint not found. Tried: {kaggle_paths}")
    
    print(f"📂 Loading checkpoint: {actual_path}")
    
    # Load checkpoint
    checkpoint = torch.load(actual_path, map_location=device)
    
    # Extract model state
    model_state = checkpoint['model_state_dict']
    
    # Remove DataParallel wrapper if present
    if list(model_state.keys())[0].startswith('module.'):
        print("🔧 Removing DataParallel wrapper...")
        model_state = {k.replace('module.', ''): v for k, v in model_state.items()}
        checkpoint['model_state_dict'] = model_state
    
    # Infer model configuration from checkpoint
    vocab_size = checkpoint.get('vocab_size', None)
    if vocab_size is None:
        # Infer from CTC head
        vocab_size = model_state['ctc_head.weight'].shape[0]
    
    # Infer architecture
    d_model = model_state['encoder.input_proj.weight'].shape[0]
    
    # Count layers
    encoder_layers = max([int(k.split('.')[2]) for k in model_state.keys() 
                         if 'encoder.layers.' in k and '.ff1.0.weight' in k]) + 1
    
    decoder_layers = max([int(k.split('.')[3]) for k in model_state.keys() 
                         if 'decoder.decoder.layers.' in k and '.linear1.weight' in k]) + 1
    
    print(f"📊 Model config: vocab_size={vocab_size}, d_model={d_model}")
    print(f"📊 Architecture: encoder_layers={encoder_layers}, decoder_layers={decoder_layers}")
    
    # Extract training info
    epoch = checkpoint.get('epoch', 0)
    optimizer_state = checkpoint.get('optimizer_state_dict', None)
    scheduler_state = checkpoint.get('scheduler_state_dict', None)
    
    print(f"📈 Last epoch: {epoch}")
    print(f"🔧 Has optimizer state: {optimizer_state is not None}")
    
    return {
        'model_state_dict': model_state,
        'optimizer_state_dict': optimizer_state,
        'scheduler_state_dict': scheduler_state,
        'epoch': epoch,
        'vocab_size': vocab_size,
        'd_model': d_model,
        'encoder_layers': encoder_layers,
        'decoder_layers': decoder_layers,
        'checkpoint': checkpoint
    }

## Step 3: Define Your Model Class

**IMPORTANT**: You need to include your model definition here or import it from your uploaded files.

In [ ]:
# Option 1: If you have the model file in your Kaggle dataset
# sys.path.append('/kaggle/input/your-model-dataset')
# from konkanivani_asr import KonkaniVaniASR

# Option 2: Define the model class directly in the notebook
# (Copy your model class definition here)

# For this example, I'll assume you have the model available
# Replace this with your actual model import/definition
class KonkaniVaniASR(nn.Module):
    def __init__(self, vocab_size, input_dim=80, d_model=512, encoder_layers=6, decoder_layers=6):
        super().__init__()
        # Your model definition here
        # This is just a placeholder - use your actual model
        pass
    
    def forward(self, x):
        # Your forward pass
        pass

## Step 4: Load Checkpoint and Create Model

In [ ]:
# Specify your checkpoint path
# This should be the name of your checkpoint file in your Kaggle input dataset
checkpoint_name = "best_model.pt"  # Change this to your checkpoint name

# Load checkpoint info
checkpoint_info = load_checkpoint_for_training_kaggle(checkpoint_name, device)

# Create model with the correct configuration
model = KonkaniVaniASR(
    vocab_size=checkpoint_info['vocab_size'],
    input_dim=80,
    d_model=checkpoint_info['d_model'],
    encoder_layers=checkpoint_info['encoder_layers'],
    decoder_layers=checkpoint_info['decoder_layers']
)

# Load the model state
model.load_state_dict(checkpoint_info['model_state_dict'])
model.to(device)

# SET TO TRAINING MODE - This is the key change!
model.train()

print(f"✅ Model loaded and set to training mode!")
print(f"📊 Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"🎯 Training mode: {model.training}")

## Step 5: Setup Optimizer and Scheduler

In [ ]:
# Create optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,  # You might want to use a lower learning rate for fine-tuning
    weight_decay=0.01
)

# Load previous optimizer state if available
if checkpoint_info['optimizer_state_dict'] is not None:
    try:
        optimizer.load_state_dict(checkpoint_info['optimizer_state_dict'])
        print("✅ Loaded optimizer state from checkpoint")
    except Exception as e:
        print(f"⚠️ Could not load optimizer state: {e}")
        print("🔄 Using fresh optimizer")

# Create scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    verbose=True
)

# Load previous scheduler state if available
if checkpoint_info['scheduler_state_dict'] is not None:
    try:
        scheduler.load_state_dict(checkpoint_info['scheduler_state_dict'])
        print("✅ Loaded scheduler state from checkpoint")
    except Exception as e:
        print(f"⚠️ Could not load scheduler state: {e}")
        print("🔄 Using fresh scheduler")

# Setup loss function
criterion = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)

print(f"📈 Will resume from epoch: {checkpoint_info['epoch'] + 1}")

## Step 6: Load Your Data

**Replace this with your actual data loading code**

In [ ]:
# This is where you load your training data
# Replace with your actual data loading code

# Example data loading (replace with your implementation)
def load_training_data():
    # Load your dataset here
    # This should return a DataLoader
    pass

# train_loader = load_training_data()
# val_loader = load_validation_data()

print("📊 Data loading setup complete")

## Step 7: Training Loop

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device, epoch):
    """Train for one epoch"""
    model.train()  # Ensure model is in training mode
    total_loss = 0
    num_batches = 0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
    
    for batch_idx, batch in enumerate(pbar):
        # Your batch processing code here
        # This depends on your data format
        
        # Example structure:
        # mel_specs, targets, input_lengths, target_lengths = batch
        # mel_specs = mel_specs.to(device)
        # targets = targets.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        # outputs = model(mel_specs)
        
        # Calculate loss
        # loss = criterion(...)
        
        # Backward pass
        # loss.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        # optimizer.step()
        
        # Update metrics
        # total_loss += loss.item()
        # num_batches += 1
        
        # pbar.set_postfix({'loss': f'{total_loss/num_batches:.4f}'})
        
        pass  # Remove this when you add your actual training code
    
    return total_loss / max(num_batches, 1)


# Training configuration
num_epochs = 10  # Number of additional epochs to train
start_epoch = checkpoint_info['epoch'] + 1

print(f"🚀 Starting training from epoch {start_epoch}")

# Training loop
for epoch in range(start_epoch, start_epoch + num_epochs):
    print(f"\n🔄 Epoch {epoch}/{start_epoch + num_epochs - 1}")
    
    # Train one epoch
    # avg_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch)
    
    # Validation (if you have validation data)
    # val_loss = validate(model, val_loader, criterion, device)
    
    # Update scheduler
    # scheduler.step(avg_loss)
    
    # Save checkpoint
    if epoch % 5 == 0:  # Save every 5 epochs
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'vocab_size': checkpoint_info['vocab_size'],
            # 'loss': avg_loss,
        }
        
        torch.save(checkpoint, f'/kaggle/working/checkpoint_epoch_{epoch}.pt')
        print(f"💾 Saved checkpoint: checkpoint_epoch_{epoch}.pt")

print("✅ Training completed!")

## Key Changes for Kaggle:

1. **Checkpoint Path**: Use `/kaggle/input/your-dataset-name/checkpoint.pt`
2. **Model Definition**: Include your model class in the notebook or upload it as a dataset
3. **Training Mode**: Call `model.train()` after loading the checkpoint
4. **Save Location**: Save new checkpoints to `/kaggle/working/`
5. **Data Paths**: Update data loading paths to Kaggle input directories

## What You Need to Change:

1. Replace `"your-checkpoint-dataset"` with your actual Kaggle dataset name
2. Add your model class definition or import
3. Add your data loading code
4. Complete the training loop with your specific loss calculation
5. Update file paths to match your Kaggle dataset structure